# Monte Carlo Pricing and Implied Volatility

Part 3 of the series. We simulate the stock price process itself, price options by averaging simulated payoffs, then invert the pricing formula to recover volatility from a price — building up to the volatility smile and surface, using a synthetic market here so the concept stands on its own before notebook 4 does the same thing with live data.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
from optionspricing import black_scholes_call, vega
from optionspricing.monte_carlo import simulate_gbm_paths, monte_carlo_price
from optionspricing.implied_vol import implied_volatility_newton

## Simulating stock prices

Solving the geometric Brownian motion SDE gives

$$ S(t) = S(0)\, e^{(\mu - \sigma^2/2)t + \sigma (X(t) - X(0))} $$

where $(\mu - \sigma^2/2)t$ is the drift (the $\sigma^2/2$ term is Ito's correction) and the second term is Brownian motion. Discretised, it's easier to simulate step by step:

$$ S(t+dt) = S(t)\, e^{(\mu - \sigma^2/2)dt + \sigma\sqrt{dt}\, Z}, \quad Z \sim N(0,1) $$

In [ ]:
S0, mu, sigma = 100, 0.08, 0.20
T, N = 1, 252  # 1 year, 252 trading days

paths = simulate_gbm_paths(S0, mu, sigma, T, n_steps=N, n_paths=100, rng=np.random.default_rng(0))

plt.figure(figsize=(10, 6))
for path in paths:
    plt.plot(path, linewidth=1)
plt.xlabel('Trading Day'); plt.ylabel('Stock Price')
plt.title('100 Simulated Geometric Brownian Motion Paths')
plt.grid(True); plt.show()

Each path is one possible evolution of the stock price under the same SDE. These simulated paths are the basis of Monte Carlo option pricing below.

## Monte Carlo option pricing

Under the risk-neutral measure we replace the real-world drift $\mu$ with $r$ (options are priced under risk-neutrality, not real-world expectations):

$$ S(T) = S(0)\, e^{(r - \sigma^2/2)T + \sigma\sqrt{T}\, Z} $$

We simulate many terminal prices, compute the payoff for each, and discount the average back to today.

In [ ]:
S, K, T, r, sigma = 100, 100, 1, 0.05, 0.20

bs_price = black_scholes_call(S, K, T, r, sigma)
mc_price = monte_carlo_price(S, K, T, r, sigma, n_simulations=100_000, rng=np.random.default_rng(0))

print(f'Black-Scholes Price : {bs_price:.6f}')
print(f'Monte Carlo Price   : {mc_price:.6f}')
print(f'Difference          : {abs(bs_price - mc_price):.6f}')

In [ ]:
simulation_sizes = [100, 500, 1000, 5000, 10000, 50000, 100000]
mc_prices = [
    monte_carlo_price(S, K, T, r, sigma, n_simulations=n, rng=np.random.default_rng(0))
    for n in simulation_sizes
]

plt.figure(figsize=(8, 5))
plt.plot(simulation_sizes, mc_prices, marker='o', label='Monte Carlo')
plt.axhline(bs_price, color='red', linestyle='--', label='Black-Scholes')
plt.xscale('log')
plt.xlabel('Number of Simulations'); plt.ylabel('Call Option Price')
plt.title('Monte Carlo Convergence'); plt.grid(True); plt.legend(); plt.show()

errors = np.abs(np.array(mc_prices) - bs_price) / bs_price * 100
plt.figure(figsize=(9, 5))
plt.plot(simulation_sizes, errors)
plt.xscale('log')
plt.xlabel('Number of Simulations'); plt.ylabel('Percentage Error (%)')
plt.title('Monte Carlo Error'); plt.grid(True); plt.show()

With few paths the estimate fluctuates noticeably due to sampling error; as the number of paths grows the estimate converges to the analytical Black-Scholes price, by the Law of Large Numbers.

## Implied volatility

Normally we know $\sigma$ and calculate a price $C$. In reality the market quotes a price $C_{market}$, and we solve $BS(\sigma) = C_{market}$ for the unknown volatility. This can't be rearranged algebraically, so we use Newton-Raphson:

$$ f(\sigma) = BS(\sigma) - C_{market}, \qquad \sigma_{n+1} = \sigma_n - f(\sigma_n)/f'(\sigma_n) $$

$f'(\sigma)$ is exactly vega, so no separate derivative code is needed — `implied_volatility_newton` in [`optionspricing.implied_vol`](../src/optionspricing/implied_vol.py) reuses `vega` directly.

In [ ]:
market_price = 10.45
iv = implied_volatility_newton(market_price, S, K, T, r)
print(f'Implied Volatility = {iv:.4f} ({100*iv:.2f}%)')

In [ ]:
def newton_history(market_price, S, K, T, r, sigma0=0.5, n_iter=10):
    history, s = [], sigma0
    for _ in range(n_iter):
        history.append(s)
        price = black_scholes_call(S, K, T, r, s)
        s = s - (price - market_price) / vega(S, K, T, r, s)
    return history

history = newton_history(market_price, S, K, T, r)

plt.figure(figsize=(8, 5))
plt.plot(range(len(history)), history, marker='o')
plt.xlabel('Iteration'); plt.ylabel('Volatility Estimate')
plt.title('Newton-Raphson Convergence'); plt.grid(True); plt.show()

## Volatility smile (synthetic)

This notebook has no live market feed — that's notebook 4. To demonstrate the smile concept and validate the solver, we build a **synthetic market**: pick a skewed volatility function $\sigma(K)$ that's higher for low strikes (the way real equity index options actually skew), price options at each strike using that *known* skewed vol, then throw away the vol and recover it purely from the resulting prices using our own Newton-Raphson solver. If the solver is correct, the recovered smile should closely retrace the input skew.

In [ ]:
strikes = np.linspace(70, 130, 40)
atm = 100

def true_skewed_vol(k):
    return 0.20 + 0.15 * np.maximum(atm - k, 0) / atm  # higher vol for low (OTM put) strikes

synthetic_prices = [black_scholes_call(S, k, T, r, true_skewed_vol(k)) for k in strikes]
recovered_ivs = [
    implied_volatility_newton(price, S, k, T, r) for price, k in zip(synthetic_prices, strikes)
]

plt.figure(figsize=(8, 5))
plt.plot(strikes, true_skewed_vol(strikes), '--', label='Input skew (ground truth)')
plt.plot(strikes, recovered_ivs, marker='o', markersize=3, label='Recovered via Newton-Raphson')
plt.xlabel('Strike Price (K)'); plt.ylabel('Implied Volatility')
plt.title('Volatility Smile: Synthetic Skew vs Recovered')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

The recovered curve overlays the input skew almost exactly, which is really a correctness check on `implied_volatility_newton` as much as it is a demo of the smile: the solver faithfully inverts whatever volatility actually generated a price, constant or not. Real markets produce a smile/skew shape like this because they price in more downside risk (crashes) than upside — see notebook 4 for a smile built from real option chains.

## Implied volatility surface (synthetic)

Extending the smile with a second axis, time to maturity, gives the full surface. We add a simple term-structure component to the synthetic vol function, then recover the surface the same way: generate prices from a known $\sigma(K, T)$, solve for IV at each grid point, and check that the recovered surface matches the input.

In [ ]:
def true_vol_surface(k, t):
    skew = 0.15 * np.maximum(atm - k, 0) / atm
    term = 0.05 * np.sqrt(t)  # longer-dated options priced with slightly more vol
    return 0.20 + skew + term

surf_strikes = np.linspace(70, 130, 25)
surf_maturities = np.linspace(0.1, 2, 25)
K_grid, T_grid = np.meshgrid(surf_strikes, surf_maturities)

recovered_surface = np.zeros_like(K_grid)
for i in range(K_grid.shape[0]):
    for j in range(K_grid.shape[1]):
        k, t = K_grid[i, j], T_grid[i, j]
        price = black_scholes_call(S, k, t, r, true_vol_surface(k, t))
        recovered_surface[i, j] = implied_volatility_newton(price, S, k, t, r)

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')
surface = ax.plot_surface(K_grid, T_grid, recovered_surface, cmap='viridis', edgecolor='none')
ax.set_xlabel('Strike Price'); ax.set_ylabel('Time to Maturity (Years)'); ax.set_zlabel('Implied Volatility')
ax.set_title('Recovered Implied Volatility Surface (Synthetic Market)')
fig.colorbar(surface, shrink=0.5, aspect=10)
plt.show()

The surface slopes up towards low strikes (skew) and towards longer maturities (term structure), exactly as built in. On a real book, this surface is the market's live map of risk pricing across every strike and maturity simultaneously — far more informative than a single volatility number. Next: [04_real_market_implied_vol.ipynb](04_real_market_implied_vol.ipynb), which builds the same kind of surface from live option chains instead of a synthetic model.